# 092 — Control estructural y edición generativa

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Tres técnicas para controlar *dónde y cómo* genera un modelo de difusión ya
entrenado:

- **ControlNet**: copia entrenable del encoder de la U-Net que recibe un mapa de
  condición (bordes, pose, profundidad); la U-Net original queda **congelada** y
  la copia se conecta a los skips del decoder mediante **zero-convolutions**
  (conv 1×1 con W = 0, b = 0 al inicio). Al arrancar, la red es idéntica al
  modelo base; el control se aprende gradualmente porque ∂(W·x)/∂W = x ≠ 0.
- **img2img (SDEdit)**: se ruidifica el latente de una imagen real hasta el paso
  `t_inicio = round(s·T)` y se denoisea desde ahí. La fuerza s ∈ [0,1] decide
  cuánta estructura del original sobrevive (s bajo = retoque, s alto = reinvención).
- **Inpainting (RePaint)**: en cada paso, la zona fuera de la máscara se fuerza a
  su valor real ruidificado y solo se genera dentro de la máscara —
  `z_{t−1} = (1−m)⊙z_conocido + m⊙z_generado` — con el contexto siempre visible.


## 🧮 Ejemplo de referencia

Con T = 50: s = 0.2 → arranca en t = 10 (10 pasos, conserva composición);
s = 0.8 → arranca en t = 40 (40 pasos, solo sobreviven rasgos globales);
s = 1.0 equivale a texto-a-imagen desde ruido puro.

Zero-convolution con W = 0, entrada x = 3.0 y gradiente entrante g = 0.5:
forward y = 0 (no perturba al modelo base), pero ∂L/∂W = g·x = 1.5 ≠ 0 →
aprende desde el primer paso; ∂L/∂x = g·W = 0 → la copia interior recibe señal
recién después de la primera actualización.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("generation", seed=92)
show(result)


## Reflexión

1. ¿Por qué inicializar las zero-convolutions en cero protege al modelo congelado en los primeros pasos y, aun así, no impide que el ControlNet aprenda? Distingue gradiente respecto a los pesos y gradiente hacia la entrada.
2. En img2img, ¿por qué la fuerza s conserva "tipo de información" (frecuencias bajas: composición, masas de color) y no un porcentaje de píxeles concretos?
3. Si el prompt dice "persona sentada" pero el mapa de pose del ControlNet muestra una figura de pie, ¿qué esperas que domine y por qué? ¿Qué papel juega el peso (conditioning scale) del ControlNet?
